# Session 3: cohort readiness

Run the cohort download, inventory, and split commands from the README first.
This notebook reads integrity results and analyzes development participants only.
Test participants receive integrity checks, but their coverage distributions remain unopened.
The split uses seed 42 and keeps participant 1066528 in development.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from sleep_classifier.cohort import load_config
from sleep_classifier.development import analyze

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/cohort.yaml").exists())
config = load_config(ROOT / "configs/cohort.yaml")
inventory = json.loads((config["output_dir"] / "inventory.json").read_text())
split = json.loads(config["split_path"].read_text())
print("Integrity-checked participants:", len(inventory["participants"]))
print("Split sizes:", {key: len(split[key]) for key in ("train", "validation", "test")})
display(
    pd.DataFrame(
        [
            {"participant": r["participant_id"], "reason": r["reasons"]}
            for r in inventory["participants"]
            if not r["eligible"]
        ]
    )
)

## Development coverage

Unscored labels (-1) are excluded from epoch summaries. Legacy N4 (4) remains a recognized scored stage.
The preparation proposal merges N3 and N4 into deep sleep later.
Each candidate requires observations from both sensors. See `configs/coverage.yaml` for exact limits.
The moderate candidate is selected as the initial baseline rule. Its thresholds are engineering assumptions, not validated accuracy optima.


In [ ]:
output = analyze(ROOT / "configs/coverage.yaml")
display(pd.read_csv(output / "aggregate-retention.csv"))
display(Image(filename=str(output / "development-coverage.png")))

## Interpretation

Compare participants as well as pooled epochs. A high overall retention rate can hide poor coverage for one participant.
Recorded step-bin hours measure interval support, not sensor quality.
Missing step coverage remains distinct from zero activity.
Rejected epochs must remain visible in evaluation coverage reports.
Keep original time indices and reset recurrent state after rejected intervals when implementing sequence preparation.
